# ECGR 4161 — GPS Trilateration Assignment

## Overview

In lecture we studied **trilateration** with **UWB (ultra-wideband) modules**:
given a handful of anchors at known positions and a measured distance to each
anchor, you can pin down a receiver's location as the point where the distance
circles (2D) or spheres (3D) intersect.

**GPS is trilateration taken to orbit** — but with two important twists that
this lab asks you to handle:

1. **The "anchors" are satellites** roughly 20,000 km overhead, and their
   positions are broadcast to the receiver in the navigation message.
2. **The receiver's clock is not perfect.** GPS signals travel *one way*
   (satellite → receiver), so the receiver measures range from signal
   *travel time*. A tiny clock error of one microsecond becomes a **300 m**
   range error. Because we cannot measure true range directly, we measure a
   **pseudorange** — the true range plus an unknown, common offset caused by
   the receiver clock bias.

That single extra unknown — the clock bias — is the heart of the GPS problem.
It is why a phone needs **four** satellites to get a 3D fix instead of three,
and it is why the equations are no longer a clean geometric intersection but a
**nonlinear least-squares** problem solved by iteration.

| | UWB trilateration (lecture) | GPS trilateration (this lab) |
|---|---|---|
| Anchors | Fixed beacons in a room | Satellites (positions broadcast) |
| Dimensions | 2D (or 3D) | 3D |
| Measurement | True distance (two-way ranging) | **Pseudorange** = range + clock offset |
| Unknowns | $x, y$ (and $z$) | $x, y, z$ **and clock bias $b$** |
| Minimum beacons | 3 (2D) | **4** |
| Solved by | Circle/sphere intersection | **Iterative least squares** |

## Your job

You will implement `solve_gps()`, a function that takes the satellite positions
and the measured pseudoranges and returns the receiver's estimated position
**and** clock bias, using iterative (linearized) least squares. The simulation
engine, the 3D visualization, and a reference solution are provided for you.

## Learning Objectives

After completing this notebook, you should be able to:

1. Explain how GPS trilateration extends UWB trilateration, and why the
   receiver clock bias adds a fourth unknown.
2. Write the **pseudorange measurement model** $\hat\rho_i = \lVert \mathbf{p} - \mathbf{s}_i \rVert + b$.
3. Build the **geometry (design) matrix** from the satellite-to-receiver
   line-of-sight unit vectors.
4. Solve the linearized normal equations to update an estimate, and iterate to
   convergence (Gauss–Newton).
5. Reason about why **four** satellites are required and how satellite geometry
   affects the quality of the fix.


## Setup

Run the following cell first. It imports the libraries used by the simulation
and the 3D visualization.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (enables 3D projection)
from IPython.display import display, clear_output

# Consistent, readable plot styling
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.4


## From UWB Trilateration to GPS

### Recap: UWB trilateration

With UWB you measure the **true distance** $d_i$ from the receiver to each anchor
$\mathbf{a}_i$. Each measurement says "the receiver lies on a sphere of radius
$d_i$ centered on anchor $i$":

$$\lVert \mathbf{p} - \mathbf{a}_i \rVert = d_i$$

Three anchors in 3D (or three circles in 2D) intersect at the receiver
position $\mathbf{p} = (x, y, z)$. Three unknowns, three equations — done.

### GPS: the receiver clock adds a fourth unknown

A GPS receiver never measures true range. It measures **travel time** of a
one-way signal and multiplies by the speed of light. But the receiver's cheap
clock is offset from GPS system time by an unknown bias $\delta t$. Expressed as
a distance, $b = c\,\delta t$, and it corrupts **every** measurement by the same
amount. What we actually observe is a **pseudorange**:

$$\rho_i = \underbrace{\lVert \mathbf{p} - \mathbf{s}_i \rVert}_{\text{true range}} + \; b$$

where $\mathbf{s}_i$ is the (known) position of satellite $i$. Now there are
**four** unknowns — $x, y, z, b$ — so we need at least **four satellites**.

### Why we need iteration

The unknown $\mathbf{p}$ sits inside a square root (the norm), so the equations
are **nonlinear** and there is no clean "intersection" formula. Instead we
**linearize around a current guess** $\mathbf{x} = [x, y, z, b]^{\mathsf T}$ and
take a step that reduces the mismatch — repeating until it converges
(the **Gauss–Newton** method).

**1. Predicted pseudorange** at the current guess:

$$\hat\rho_i = \lVert \mathbf{p} - \mathbf{s}_i \rVert + b$$

**2. Residual** (measured minus predicted):

$$\Delta\rho_i = \rho_i - \hat\rho_i$$

**3. Geometry (design) matrix.** The partial derivatives of $\hat\rho_i$ form
the line-of-sight unit vector from the satellite toward the current position,
plus a $1$ for the clock term:

$$
\mathbf{e}_i = \frac{\mathbf{p} - \mathbf{s}_i}{\lVert \mathbf{p} - \mathbf{s}_i \rVert},
\qquad
G = \begin{bmatrix}
 e_{1x} & e_{1y} & e_{1z} & 1 \\
 e_{2x} & e_{2y} & e_{2z} & 1 \\
 \vdots & \vdots & \vdots & \vdots \\
 e_{Nx} & e_{Ny} & e_{Nz} & 1
\end{bmatrix}
$$

**4. Least-squares update.** Solve the (generally over-determined) linear system
$G\,\Delta\mathbf{x} = \Delta\boldsymbol{\rho}$ for the correction
$\Delta\mathbf{x} = [\Delta x, \Delta y, \Delta z, \Delta b]^{\mathsf T}$:

$$\Delta\mathbf{x} = (G^{\mathsf T} G)^{-1} G^{\mathsf T}\,\Delta\boldsymbol{\rho}$$

(In code, `np.linalg.lstsq(G, delta_rho)` does exactly this, more robustly.)

**5. Apply and repeat.** Update $\mathbf{x} \leftarrow \mathbf{x} + \Delta\mathbf{x}$
and loop back to step 1. Starting from the Earth's center
$\mathbf{x} = [0, 0, 0, 0]$, this converges in a handful of iterations.


## Your Task: Implement `solve_gps()`

Complete the function `solve_gps(sat_positions, pseudoranges, initial_guess=None, num_iterations=15)`.

You are given:

- `sat_positions` — an array of shape `(N, 3)`: the `(x, y, z)` position of each
  of the `N` satellites (kilometers).
- `pseudoranges` — an array of shape `(N,)`: the measured pseudorange to each
  satellite (kilometers), i.e. true range **plus** the common clock bias.

You must return a length-4 NumPy array `[x, y, z, b]`:

- `x, y, z` — the estimated receiver position (km)
- `b` — the estimated clock bias expressed as a distance (km)

Implement the **Gauss–Newton iteration** from the theory section. Each iteration:

| Step | What to compute |
|---|---|
| 1 | Split the current estimate into position `p = x[:3]` and bias `b = x[3]`. |
| 2 | Geometric ranges `r_i = ‖p − s_i‖` to every satellite. |
| 3 | Predicted pseudoranges `ρ̂_i = r_i + b`. |
| 4 | Residuals `Δρ_i = ρ_i(measured) − ρ̂_i`. |
| 5 | Geometry matrix `G` with rows `[ (p − s_i)/r_i , 1 ]` (line-of-sight unit vector + a 1). |
| 6 | Correction `Δx = lstsq(G, Δρ)`. |
| 7 | Update `x ← x + Δx`. Stop early if `‖Δx‖` is tiny. |

### Hints

- If `initial_guess is None`, start from `np.zeros(4)` (Earth's center, zero bias).
- `np.linalg.norm(diffs, axis=1)` gives all `N` ranges at once when
  `diffs = p - sat_positions` has shape `(N, 3)`.
- Build `G` as an `(N, 4)` array: columns 0–2 are `diffs / r[:, None]`, column 3 is all ones.
- Use `np.linalg.lstsq(G, delta_rho, rcond=None)[0]` to solve for `Δx`.
- Vectorize with NumPy — no Python loop over satellites is needed inside an iteration.

### Expected behavior

| Scenario | Expected result |
|---|---|
| 4 satellites, no noise | Recovers the true position and bias essentially exactly |
| 6+ satellites, no noise | Same — extra satellites are handled by least squares |
| Satellites with measurement noise | A best-fit estimate close to the true position |
| Fewer than 4 satellites | Under-determined — not required to handle; 4 is the minimum |


In [ ]:
def solve_gps(sat_positions, pseudoranges, initial_guess=None, num_iterations=15):
    """
    Estimate a GPS receiver's position and clock bias by iterative least squares.

    Parameters
    ----------
    sat_positions : array-like, shape (N, 3)
        Known (x, y, z) position of each satellite, in kilometers.
    pseudoranges : array-like, shape (N,)
        Measured pseudorange to each satellite, in kilometers.
        A pseudorange is the true geometric range PLUS the common receiver
        clock bias b (expressed as a distance).
    initial_guess : array-like, shape (4,), optional
        Starting estimate [x, y, z, b]. If None, start at Earth's center with
        zero clock bias: np.zeros(4).
    num_iterations : int
        Maximum number of Gauss-Newton iterations.

    Returns
    -------
    numpy.ndarray, shape (4,)
        The estimated state [x, y, z, b]:
          x, y, z -- receiver position (km)
          b       -- receiver clock bias as a distance (km)

    Notes
    -----
    Implement the Gauss-Newton loop described in the theory section:
        1. p = x[:3], b = x[3]
        2. ranges r_i = || p - s_i ||
        3. predicted pseudorange rho_hat_i = r_i + b
        4. residual d_rho_i = rho_i - rho_hat_i
        5. geometry matrix G row i = [ (p - s_i) / r_i , 1 ]
        6. correction dx = lstsq(G, d_rho)
        7. x = x + dx   (stop early if ||dx|| is tiny)
    """
    sat_positions = np.asarray(sat_positions, dtype=float)
    pseudoranges = np.asarray(pseudoranges, dtype=float)

    # TODO: Initialize the estimate x (length-4: [x, y, z, b]).
    #   Use initial_guess if given, otherwise np.zeros(4).

    # TODO: Repeat num_iterations times:
    #   Step 1 -- split x into position p (x[:3]) and bias b (x[3])
    #   Step 2 -- diffs = p - sat_positions            # shape (N, 3)
    #             r = np.linalg.norm(diffs, axis=1)    # shape (N,)
    #   Step 3 -- predicted = r + b
    #   Step 4 -- delta_rho = pseudoranges - predicted
    #   Step 5 -- build G of shape (N, 4):
    #                 G[:, :3] = diffs / r[:, None]
    #                 G[:, 3]  = 1.0
    #   Step 6 -- dx = np.linalg.lstsq(G, delta_rho, rcond=None)[0]
    #   Step 7 -- x = x + dx
    #             break early if np.linalg.norm(dx) < 1e-9

    # TODO: return the final estimate x

    pass  # Replace this line with your implementation


## Simulation Engine

Run this cell after implementing `solve_gps()`. It defines the constants, the
scenario generator, a reference solver, and the 3D visualization. **You do not
need to modify any code in this cell.**

All distances are in **kilometers**. For reference, the real GPS constellation
orbits at a radius of about 26,600 km from Earth's center, and the Earth's mean
radius is about 6,371 km.


In [ ]:
# ── Physical constants (kilometers) ────────────────────────────────────────────
EARTH_RADIUS = 6371.0        # mean Earth radius (km)
SAT_ORBIT_RADIUS = 26571.0   # GPS orbit radius from Earth's center (km)


# ── Scenario generator ─────────────────────────────────────────────────────────

def generate_gps_scenario(num_satellites=6, clock_bias=50.0, noise_std=0.0, seed=0):
    """
    Build a synthetic GPS scenario.

    Places a receiver on the Earth's surface and a set of satellites in the sky
    above it, then computes the pseudorange each satellite would measure
    (true range + common clock bias + optional Gaussian noise).

    Returns
    -------
    sat_positions : (N, 3) ndarray   satellite positions (km)
    pseudoranges  : (N,)   ndarray   measured pseudoranges (km)
    true_pos      : (3,)   ndarray   true receiver position (km)
    true_bias     : float            true clock bias as a distance (km)
    """
    rng = np.random.default_rng(seed)

    # True receiver somewhere on the Earth's surface (northern-ish hemisphere)
    lat = np.radians(rng.uniform(20.0, 60.0))
    lon = np.radians(rng.uniform(-120.0, -60.0))
    r_hat = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])
    true_pos = EARTH_RADIUS * r_hat

    # Satellites scattered across the sky above the receiver (above the horizon)
    sats = []
    while len(sats) < num_satellites:
        d = r_hat + rng.normal(0.0, 0.8, 3)
        d = d / np.linalg.norm(d)
        if np.dot(d, r_hat) > 0.15:            # keep it above the local horizon
            sats.append(d * SAT_ORBIT_RADIUS)
    sat_positions = np.array(sats)

    true_ranges = np.linalg.norm(true_pos - sat_positions, axis=1)
    noise = rng.normal(0.0, noise_std, num_satellites) if noise_std > 0 else 0.0
    pseudoranges = true_ranges + clock_bias + noise

    return sat_positions, pseudoranges, true_pos, float(clock_bias)


# ── Reference solver (used to teach convergence and to grade) ──────────────────

def _reference_solve_gps(sat_positions, pseudoranges, initial_guess=None,
                         num_iterations=15, return_history=False):
    """Correct Gauss-Newton GPS solver. Used internally as the reference."""
    sat_positions = np.asarray(sat_positions, dtype=float)
    pseudoranges = np.asarray(pseudoranges, dtype=float)

    x = np.zeros(4) if initial_guess is None else np.array(initial_guess, float)
    history = [x.copy()]

    for _ in range(num_iterations):
        p, b = x[:3], x[3]
        diffs = p - sat_positions
        r = np.linalg.norm(diffs, axis=1)
        predicted = r + b
        delta_rho = pseudoranges - predicted

        G = np.empty((len(sat_positions), 4))
        G[:, :3] = diffs / r[:, None]
        G[:, 3] = 1.0

        dx = np.linalg.lstsq(G, delta_rho, rcond=None)[0]
        x = x + dx
        history.append(x.copy())
        if np.linalg.norm(dx) < 1e-9:
            break

    if return_history:
        return x, np.array(history)
    return x


# ── 3D scene drawing ───────────────────────────────────────────────────────────

def _draw_earth(ax):
    """Draw a translucent Earth sphere."""
    u = np.linspace(0, 2 * np.pi, 36)
    v = np.linspace(0, np.pi, 18)
    xs = EARTH_RADIUS * np.outer(np.cos(u), np.sin(v))
    ys = EARTH_RADIUS * np.outer(np.sin(u), np.sin(v))
    zs = EARTH_RADIUS * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_surface(xs, ys, zs, color="lightblue", alpha=0.20,
                    linewidth=0, zorder=0)


def _draw_scene(ax, sat_positions, true_pos, est_pos):
    """Draw satellites, lines of sight, the true position, and the estimate."""
    ax.clear()
    _draw_earth(ax)

    # Satellites and their lines of sight to the true receiver
    ax.scatter(sat_positions[:, 0], sat_positions[:, 1], sat_positions[:, 2],
               color="darkorange", s=60, depthshade=True, label="Satellites",
               zorder=5)
    for s in sat_positions:
        ax.plot([s[0], true_pos[0]], [s[1], true_pos[1]], [s[2], true_pos[2]],
                color="gray", linewidth=0.8, alpha=0.5, zorder=1)

    # True receiver position
    ax.scatter([true_pos[0]], [true_pos[1]], [true_pos[2]],
               color="green", s=90, marker="*", label="True position", zorder=6)

    # Current estimate
    ax.scatter([est_pos[0]], [est_pos[1]], [est_pos[2]],
               color="crimson", s=70, marker="X", label="Estimate", zorder=7)

    lim = SAT_ORBIT_RADIUS * 1.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_zlabel("z (km)")
    ax.legend(loc="upper left", fontsize=8)


# ── Visualization + student check ──────────────────────────────────────────────

def visualize_gps_solution(scenario, animate=True, fps=2):
    """
    Run the student's solve_gps() on a scenario and visualize the result.

    The animation (using the reference solver's iteration history) shows the
    estimate converging from Earth's center toward the receiver, alongside a
    convergence curve. The final frame plots the student's own solution and a
    match verdict.
    """
    sat_positions, pseudoranges, true_pos, true_bias = scenario

    # Reference iteration history (for the convergence story)
    ref_state, history = _reference_solve_gps(
        sat_positions, pseudoranges, return_history=True
    )
    pos_err_hist = np.linalg.norm(history[:, :3] - true_pos, axis=1)

    # Student's solution (this is what gets graded)
    student_error = None
    try:
        student_state = np.asarray(solve_gps(sat_positions, pseudoranges), float)
        if student_state.shape != (4,):
            raise ValueError(
                f"solve_gps must return 4 numbers [x, y, z, b]; "
                f"got shape {student_state.shape}"
            )
    except Exception as exc:  # noqa: BLE001
        student_error = exc

    fig = plt.figure(figsize=(13, 6))
    gs = gridspec.GridSpec(2, 2, width_ratios=[1.4, 1.0], height_ratios=[1, 1])
    ax3d = fig.add_subplot(gs[:, 0], projection="3d")
    ax_err = fig.add_subplot(gs[0, 1])
    ax_bias = fig.add_subplot(gs[1, 1])

    def _draw_convergence(k):
        ax_err.clear()
        ax_err.plot(range(k + 1), pos_err_hist[:k + 1], "-o",
                    color="crimson", markersize=4)
        ax_err.set_yscale("log")
        ax_err.set_xlabel("iteration")
        ax_err.set_ylabel("position error (km)")
        ax_err.set_title("Convergence of position error", fontsize=10)
        ax_err.grid(True, which="both", linestyle="--", alpha=0.4)

        ax_bias.clear()
        ax_bias.plot(range(k + 1), history[:k + 1, 3], "-o",
                     color="steelblue", markersize=4, label="estimated b")
        ax_bias.axhline(true_bias, color="green", linestyle="--",
                        linewidth=1.2, label="true b")
        ax_bias.set_xlabel("iteration")
        ax_bias.set_ylabel("clock bias (km)")
        ax_bias.set_title("Clock-bias estimate", fontsize=10)
        ax_bias.legend(fontsize=8)
        ax_bias.grid(True, linestyle="--", alpha=0.4)

    if animate:
        for k in range(len(history)):
            _draw_scene(ax3d, sat_positions, true_pos, history[k, :3])
            ax3d.set_title(
                f"GPS trilateration — least-squares iteration {k}",
                fontsize=11, weight="bold"
            )
            _draw_convergence(k)
            clear_output(wait=True)
            display(fig)
            time.sleep(1.0 / fps)

    # ── Final frame: show the student's own solution + verdict ────────────────
    _draw_convergence(len(history) - 1)

    if student_error is not None:
        _draw_scene(ax3d, sat_positions, true_pos, np.zeros(3))
        ax3d.set_title(f"Error in your solve_gps(): {student_error}",
                       fontsize=10, color="crimson", weight="bold")
        clear_output(wait=True)
        display(fig)
        plt.close(fig)
        print(f"⚠ Your solve_gps() raised an error: {student_error}")
        return

    pos_err = float(np.linalg.norm(student_state[:3] - true_pos))
    bias_err = float(abs(student_state[3] - true_bias))
    # A correct solver matches the reference essentially exactly.
    match = np.allclose(student_state, ref_state, atol=1e-3)

    _draw_scene(ax3d, sat_positions, true_pos, student_state[:3])
    verdict = ("✔ Matches reference solution" if match
               else "✘ Differs from reference solution")
    color = "green" if match else "crimson"
    ax3d.set_title(
        f"Your solve_gps() result\n{verdict}",
        fontsize=11, weight="bold", color=color
    )
    ax_err.plot(len(history) - 1, pos_err, "X", color="black", markersize=9,
                label="your error")
    ax_err.legend(fontsize=8)

    clear_output(wait=True)
    display(fig)
    plt.close(fig)

    print("── Your solution ─────────────────────────────")
    print(f"  Estimated position : ({student_state[0]:.3f}, "
          f"{student_state[1]:.3f}, {student_state[2]:.3f}) km")
    print(f"  True position      : ({true_pos[0]:.3f}, "
          f"{true_pos[1]:.3f}, {true_pos[2]:.3f}) km")
    print(f"  Position error     : {pos_err:.4f} km  ({pos_err * 1000:.1f} m)")
    print(f"  Estimated bias     : {student_state[3]:.3f} km   "
          f"(true {true_bias:.3f} km, error {bias_err:.4f} km)")
    print("──────────────────────────────────────────────")
    print(verdict.replace("✔", "PASS —").replace("✘", "FAIL —"))


## Concept Bridge: UWB Circles → GPS Least Squares

Before solving the GPS problem, run the cell below. On the **left** is the UWB
trilateration you saw in lecture: three anchors, three exact distance circles,
one intersection point. On the **right** is what happens when every distance is
corrupted by the **same** unknown clock bias (as in GPS): the circles no longer
meet at a single point — they enclose a region. Adding satellites and solving
for that shared bias as a fourth unknown is exactly what `solve_gps()` does.


In [ ]:
def plot_uwb_vs_gps_recap():
    """Side-by-side 2D picture: exact UWB circles vs. clock-biased 'pseudorange' circles."""
    anchors = np.array([[0.0, 0.0], [10.0, 0.0], [4.0, 8.0]])
    receiver = np.array([4.0, 3.0])
    true_dists = np.linalg.norm(anchors - receiver, axis=1)
    bias = 2.0  # common range offset added by an imperfect clock (arbitrary units)

    theta = np.linspace(0, 2 * np.pi, 200)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

    for ax, dists, title, note in (
        (axes[0], true_dists,
         "UWB: true distances → single intersection",
         "circles meet at the receiver"),
        (axes[1], true_dists + bias,
         "GPS: pseudoranges (range + clock bias)",
         "biased circles enclose a region, not a point"),
    ):
        for a, d in zip(anchors, dists):
            ax.plot(a[0] + d * np.cos(theta), a[1] + d * np.sin(theta),
                    linewidth=1.6, alpha=0.8)
        ax.scatter(anchors[:, 0], anchors[:, 1], color="darkorange",
                   s=70, zorder=5, label="Anchors / satellites")
        ax.scatter(*receiver, color="green", marker="*", s=160, zorder=6,
                   label="True receiver")
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.text(0.5, -0.14, note, transform=ax.transAxes, ha="center",
                fontsize=9, style="italic", color="dimgray")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()


plot_uwb_vs_gps_recap()


## Try Your Function

Run the cell below after implementing `solve_gps()`.

The 3D view animates the estimate converging from the Earth's center (iteration 0)
toward the true receiver over a handful of least-squares iterations. The panels
on the right show the **position error** shrinking (log scale) and the
**clock-bias estimate** locking onto its true value. The final frame plots
**your** function's solution and prints a match verdict.

Experiment with the scenario parameters:

| Parameter | Try | Effect |
|---|---|---|
| `num_satellites` | `4`, `6`, `10` | 4 is the minimum; more improves robustness |
| `clock_bias` | `0.0`, `50.0`, `200.0` | the common offset your solver must recover |
| `noise_std` | `0.0`, `0.01`, `0.05` | measurement noise (km); watch the residual error |
| `seed` | any integer | changes the receiver location and satellite geometry |


In [ ]:
# ── Scenario parameters ────────────────────────────────────────────────────────
num_satellites = 6      # number of visible satellites (minimum 4)
clock_bias     = 50.0   # true receiver clock bias as a distance (km)
noise_std      = 0.0    # pseudorange measurement noise std-dev (km); try 0.02
seed           = 0      # change for a different receiver / satellite geometry

scenario = generate_gps_scenario(
    num_satellites=num_satellites,
    clock_bias=clock_bias,
    noise_std=noise_std,
    seed=seed,
)

visualize_gps_solution(scenario, animate=True, fps=2)


## Self-Checks

Run this cell to automatically verify your implementation. Each test builds a
scenario, calls your `solve_gps()`, and compares your estimated position and
clock bias against the reference solver. Noise-free cases must match to within
1 meter; the noisy case allows a looser tolerance.


In [ ]:
def _check_scenario(desc, scenario, pos_tol, bias_tol):
    sat_positions, pseudoranges, true_pos, true_bias = scenario
    try:
        est = np.asarray(solve_gps(sat_positions, pseudoranges), float)
    except Exception as exc:  # noqa: BLE001
        print(f"FAIL: {desc}")
        print(f"  Error: {exc}")
        return False

    if est.shape != (4,):
        print(f"FAIL: {desc}")
        print(f"  solve_gps must return 4 numbers [x, y, z, b]; got shape {est.shape}")
        return False

    pos_err = float(np.linalg.norm(est[:3] - true_pos))
    bias_err = float(abs(est[3] - true_bias))
    if pos_err <= pos_tol and bias_err <= bias_tol:
        print(f"PASS: {desc}  (pos err {pos_err * 1000:.2f} m, bias err {bias_err * 1000:.2f} m)")
        return True

    print(f"FAIL: {desc}")
    print(f"  Position error: {pos_err:.5f} km (tol {pos_tol:.5f} km)")
    print(f"  Bias error:     {bias_err:.5f} km (tol {bias_tol:.5f} km)")
    return False


# (description, scenario kwargs, position tolerance km, bias tolerance km)
_tests = [
    ("4 satellites, no bias, no noise",
     dict(num_satellites=4, clock_bias=0.0,   noise_std=0.0, seed=1), 1e-3, 1e-3),
    ("4 satellites (minimum), with bias",
     dict(num_satellites=4, clock_bias=50.0,  noise_std=0.0, seed=2), 1e-3, 1e-3),
    ("6 satellites, large bias",
     dict(num_satellites=6, clock_bias=200.0, noise_std=0.0, seed=3), 1e-3, 1e-3),
    ("10 satellites, over-determined",
     dict(num_satellites=10, clock_bias=75.0, noise_std=0.0, seed=4), 1e-3, 1e-3),
    ("different geometry (seed 7)",
     dict(num_satellites=6, clock_bias=30.0,  noise_std=0.0, seed=7), 1e-3, 1e-3),
    ("8 satellites with measurement noise",
     dict(num_satellites=8, clock_bias=60.0,  noise_std=0.02, seed=5), 0.5, 0.5),
]

all_pass = True
for desc, kwargs, pos_tol, bias_tol in _tests:
    scenario = generate_gps_scenario(**kwargs)
    if not _check_scenario(desc, scenario, pos_tol, bias_tol):
        all_pass = False

print()
if all_pass:
    print("All tests passed! Great work.")
else:
    print("Some tests failed. Re-read the Gauss-Newton steps in 'Your Task' and "
          "check your geometry matrix and least-squares update.")
